# 🐕 Dogbot Fleet Management & Flashing
This notebook helps manage the fleet of 50 dogbots. It keeps track of which ones have been flashed, what their `mDNS` hostnames are, and provides the exact ESP-IDF command to flash the next available robot.

## 1. Initialize Fleet Manifest
Run this cell **ONCE** to generate the initial CSV file containing all 50 dogbots. If the file already exists, it will load it instead.

In [ ]:
import pandas as pd
import os
from IPython.display import Markdown, display

CSV_FILE = 'dogbot_fleet.csv'

if not os.path.exists(CSV_FILE):
    dogs = []
    for i in range(1, 51):
        dogs.append({
            'id': i,
            'hostname': f'dogbot{i}',
            'full_mdns': f'dogbot{i}.local',
            'mac_address': '',
            'ip_address': '',
            'location': 'Storage',
            'status': 'unflashed',
            'last_seen': '',
            'firmware_version': '0.0.1',
            'notes': ''
        })
    df = pd.DataFrame(dogs)
    df.to_csv(CSV_FILE, index=False)
    print(f"Created manifest for {len(df)} dogbots 🐕")
else:
    df = pd.read_csv(CSV_FILE)
    print(f"Loaded existing manifest with {len(df)} dogbots 🐕")

## 2. Flash the Next Available Dogbot
Run this cell to find the first robot marked as `unflashed` and get its flash command. Connect the dogbot via USB before running the command in your terminal!

In [ ]:
# Load the latest dataframe
df = pd.read_csv(CSV_FILE)
unflashed = df[df['status'] == 'unflashed']

if len(unflashed) > 0:
    next_bot = unflashed.iloc[0]
    bot_id = next_bot['id']
    cmd = f"idf.py build -DDEVICE_NUMBER={bot_id} && idf.py flash"
    
    display(Markdown(f"### 🤖 Next Robot: **Robot #{bot_id}** (`{next_bot['full_mdns']}`)"))
    display(Markdown("Run the following command in your terminal to build and flash:"))
    display(Markdown(f"```bash\n{cmd}\n```"))
else:
    display(Markdown("### 🎉 All 50 dogbots have been flashed!"))

## 3. Mark as Flashed
Once the flashing is successfully completed, run this cell to update the status in the CSV file so you can move on to the next one.

In [ ]:
if len(unflashed) > 0:
    bot_id = next_bot['id']
    df.loc[df['id'] == bot_id, 'status'] = 'flashed'
    df.to_csv(CSV_FILE, index=False)
    display(Markdown(f"\u2705 **Robot #{bot_id}** marked as flashed! Proceed to the next one."))
else:
    print("No unflashed robots to update.")